# Completude e Inconsistências de Preenchimento

---

## Objetivo

Calcular o **percentual de completude** de variáveis importantes, identificar **inconsistências** de preenchimento e explicar o impacto desses problemas nos indicadores e dashboards do projeto.

### Tipos de ausência

É fundamental diferenciar dois tipos de dado faltante antes de qualquer tratamento:

| Tipo | Definição | Exemplo |
|:---|:---|:---|
| **Condicional** | Ausência esperada e estrutural, decorrente da lógica do dado | `prorrogada_ate` só existe se a pesquisa foi prorrogada; `linha_matriz` só existe para questões de matriz |
| **A investigar** | Ausência inesperada que pode indicar falha de preenchimento ou processo | `complexidade` ausente em 75% das pesquisas; `data_solicitacao` ausente em 4 pesquisas |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})

BASE_PATH = '../data/raw/'

pesquisas    = pd.read_csv(BASE_PATH + 'pesquisas.csv',    sep=';')
empresas     = pd.read_csv(BASE_PATH + 'empresas.csv',     sep=';')
clientes     = pd.read_csv(BASE_PATH + 'clientes.csv',     sep=';')
respondentes = pd.read_csv(BASE_PATH + 'respondentes.csv', sep=';')
questoes     = pd.read_csv(BASE_PATH + 'questoes.csv',     sep=';')
respostas    = pd.read_csv(BASE_PATH + 'respostas.csv',    sep=';')

bases = {
    'pesquisas':    pesquisas,
    'empresas':     empresas,
    'clientes':     clientes,
    'respondentes': respondentes,
    'questoes':     questoes,
    'respostas':    respostas,
}

print('Bases carregadas.')

---

## 1. Inventário geral: registros, colunas e nulos por base


In [ ]:
def inventario(nome, df):
    n_linhas, n_cols = df.shape
    total_celulas = n_linhas * n_cols
    total_nulos   = df.isnull().sum().sum()
    completude    = 100 * (1 - total_nulos / total_celulas)
    cols_com_nulo = (df.isnull().sum() > 0).sum()
    return {
        'Base': nome,
        'Registros': n_linhas,
        'Colunas': n_cols,
        'Células totais': total_celulas,
        'Células nulas': total_nulos,
        'Completude geral (%)': round(completude, 1),
        'Colunas c/ nulo': cols_com_nulo,
    }

inv = pd.DataFrame([inventario(n, df) for n, df in bases.items()])
print('=== Inventário geral ===')
print(inv.to_string(index=False))

---

## 2. Completude por campo — pesquisas.csv

Esta é a tabela mais crítica para os KPIs operacionais. Campos como `data_solicitacao`, `complexidade` e `participantes` alimentam diretamente os indicadores I01–I06 do Data Model Canvas.


In [ ]:
def completude_por_campo(df, nome_base):
    n = len(df)
    nulos = df.isnull().sum()
    result = pd.DataFrame({
        'Campo': nulos.index,
        'Nulos': nulos.values,
        'Completude (%)': ((n - nulos.values) / n * 100).round(1),
    })
    result = result[result['Nulos'] > 0].sort_values('Completude (%)')
    return result

# ── Classificação manual das ausências em pesquisas.csv ─────────────────────
natureza_pesquisas = {
    'data_aprov':                                   'Condicional',
    'prorrogada_ate':                               'Condicional',
    'complexidade':                                 'A investigar',
    'tempo_preenchimento':                          'A investigar',
    'objetivos':                                    'A investigar',
    'part_mercado':                                 'A investigar',
    'dias para tabulação':                          'Condicional',
    'dias_uteis_para_entrega':                      'Condicional',
    'dias entre a solicitação e a divulgação':      'Condicional',
    'data_entrega':                                 'Condicional',
    'participantes':                                'A investigar',
    'data_divulgacao':                              'Condicional',
    'data_limite':                                  'Condicional',
    'data_solicitacao':                             'A investigar',
    'dia da semana de lançamento':                  'Condicional',
    'dias úteis aberta':                            'Condicional',
}

comp_pesq = completude_por_campo(pesquisas.drop(columns=[c for c in pesquisas.columns if 'Unnamed' in str(c)]), 'pesquisas')
comp_pesq['Natureza'] = comp_pesq['Campo'].map(natureza_pesquisas).fillna('Condicional')

print('=== Completude por campo — pesquisas.csv (apenas campos com nulos) ===')
print(comp_pesq[['Campo', 'Nulos', 'Completude (%)', 'Natureza']].to_string(index=False))

In [ ]:
# Visualização: completude com destaque por natureza
cores_nat = {'Condicional': '#4a90c4', 'A investigar': '#e07b39'}

fig, ax = plt.subplots(figsize=(10, 5))
cores_barras = [cores_nat[n] for n in comp_pesq['Natureza']]
bars = ax.barh(comp_pesq['Campo'], comp_pesq['Completude (%)'], color=cores_barras, height=0.65)

for bar, val in zip(bars, comp_pesq['Completude (%)']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.0f}%', va='center', fontsize=8.5)

ax.axvline(100, color='#555', linewidth=0.7, linestyle='--', alpha=0.5)
ax.set_xlabel('Completude (%)')
ax.set_xlim(0, 110)
ax.set_title('Completude por campo — pesquisas.csv', fontweight='bold')

patches = [mpatches.Patch(color=v, label=k) for k, v in cores_nat.items()]
ax.legend(handles=patches, loc='lower right')
plt.tight_layout()
plt.savefig('fig_02_completude_pesquisas.png', bbox_inches='tight')
plt.show()

---

## 3. Completude — demais bases


In [ ]:
# Natureza das ausências nas demais bases (mapeamento manual)
natureza_outras = {
    'empresas': {
        'Nacionalidade': 'A investigar', 'País': 'A investigar',
        'Tipo de Associado': 'Condicional', 'Estado': 'A investigar',
        'Cidade': 'A investigar', 'Associado': 'A investigar',
    },
    'clientes': {
        'cargo': 'A investigar', 'departamento': 'A investigar',
        'id_empresa': 'A investigar', 'e-mail': 'A investigar',
    },
    'respondentes': {
        'hora_inicio': 'Condicional', 'hora_termino': 'Condicional',
        'id_cliente': 'A investigar', 'market_share': 'A investigar',
    },
    'questoes': {
        'observacoes': 'Condicional', 'linhas_matriz': 'Condicional',
        'colunas_matriz': 'Condicional', 'alternativas': 'Condicional',
        'qtd_alternativas': 'Condicional',
    },
    'respostas': {
        'linha_matriz': 'Condicional', 'coluna_matriz': 'Condicional',
        'alternativa': 'Condicional', 'valor': 'Condicional',
        'id_empresa': 'A investigar',
    },
}

for nome in ['empresas', 'clientes', 'respondentes', 'questoes', 'respostas']:
    df = bases[nome]
    comp = completude_por_campo(df, nome)
    if len(comp) == 0:
        print(f'\n{nome}.csv — nenhum campo com nulo.')
        continue
    nat_map = natureza_outras.get(nome, {})
    comp['Natureza'] = comp['Campo'].map(nat_map).fillna('Condicional')
    print(f'\n=== {nome}.csv ===')
    print(comp[['Campo', 'Nulos', 'Completude (%)', 'Natureza']].to_string(index=False))

---

## 4. Verificação de duplicatas


In [ ]:
chaves_primarias = {
    'pesquisas':    ['id'],
    'empresas':     ['ID'],
    'clientes':     ['id_cliente'],
    'questoes':     ['id_pergunta'],
    'respostas':    ['id_resposta'],
    # respondentes não tem chave primária simples declarada — analisamos a combinação
}

print('=== Verificação de duplicatas por chave primária ===')
for nome, chave in chaves_primarias.items():
    df = bases[nome]
    n_total = len(df)
    n_unicos = df[chave].drop_duplicates().shape[0]
    dups = n_total - n_unicos
    status = '✓ Sem duplicatas' if dups == 0 else f'⚠ {dups} duplicatas'
    print(f'  {nome:15s} | Chave: {str(chave):30s} | Total: {n_total:6d} | Únicos: {n_unicos:6d} | {status}')

# Verificação especial: duplicatas de linha completa
print()
print('=== Verificação de linhas completamente duplicadas ===')
for nome, df in bases.items():
    n_dup_linhas = df.duplicated().sum()
    status = '✓ Sem duplicatas' if n_dup_linhas == 0 else f'⚠ {n_dup_linhas} linhas duplicadas'
    print(f'  {nome:15s} | {status}')

---

## 5. Inconsistências de preenchimento identificadas


In [ ]:
print('=== Inconsistência 1: variações de capitalização em empresas.Associado ===')
print(empresas['Associado'].value_counts(dropna=False))
print()
print('→ "Sim" e "sim" representam o mesmo valor mas aparecem como categorias distintas.')
print('  Solução: normalizar para maiúscula na camada de ingestão.')

In [ ]:
print('=== Inconsistência 2: variações em empresas.País ===')
print(empresas['País'].value_counts(dropna=False))
print()
print('→ "Brasil", "BRASIL" e "brasil" são o mesmo país — inconsistência de capitalização.')

In [ ]:
print('=== Inconsistência 3: sentinela "s/i" em clientes.id_cliente ===')
print(clientes[clientes['id_cliente'] == 's/i'])
print()
print('→ "s/i" (sem informação) misturado a IDs numéricos. Deve ser tratado como nulo semântico,')
print('  não como um ID real. Excluir de operações de chave.')

In [ ]:
print('=== Inconsistência 4: percentuais armazenados como texto ===')
for col, df, nome in [('part_mercado', pesquisas, 'pesquisas'), 
                       ('% de mercado', empresas, 'empresas'),
                       ('market_share', respondentes, 'respondentes')]:
    amostra = df[col].dropna().head(5).tolist()
    print(f'  {nome}.{col}: {amostra}')
    
print()
print('→ Todos são numéricos armazenados como texto (vírgula decimal, símbolo %).')
print('  Operações de média ou soma falharão sem conversão prévia.')
print()

# Demonstração da conversão correta
empresas['pct_mercado_num'] = (
    empresas['% de mercado']
    .str.replace('%', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)
print('Após conversão — empresas.% de mercado:')
print(empresas['pct_mercado_num'].describe().round(4))

In [ ]:
print('=== Inconsistência 5: tipo incompatível de id_cliente entre bases ===')
print(f'  clientes.id_cliente   dtype: {clientes["id_cliente"].dtype}  | exemplo: {clientes["id_cliente"].iloc[1]}')
print(f'  respondentes.id_cliente dtype: {respondentes["id_cliente"].dtype} | exemplo: {respondentes["id_cliente"].dropna().iloc[0]}')
print()
print('→ Tipos técnicos diferentes dificultam o JOIN entre as tabelas.')
print('  Requer regra explícita de normalização de tipo antes de qualquer vínculo.')

---

## 6. Impacto das inconsistências nos indicadores e dashboards


In [ ]:
impactos = [
    {
        'Problema': 'complexidade ausente em 75,4% das pesquisas',
        'Indicadores afetados': 'I02, I04, I06',
        'Dashboard afetado': 'D01 — Operação e portfólio',
        'Risco': 'Filtros por complexidade retornam conjuntos incompletos; análise de tempo de ciclo por nível perde representatividade.'
    },
    {
        'Problema': 'participantes ausente em 8,5% das pesquisas',
        'Indicadores afetados': 'I01, I07',
        'Dashboard afetado': 'D01, D02',
        'Risco': 'Impossível calcular média de participantes por período sem excluir esses registros. Denominador de adesão fica indisponível para essas pesquisas.'
    },
    {
        'Problema': 'part_mercado / % de mercado armazenados como texto',
        'Indicadores afetados': 'I09',
        'Dashboard afetado': 'D02 — Adesão e representatividade',
        'Risco': 'Operações aritméticas retornam erro ou resultado incorreto (ordenação lexicográfica). Indicador de representatividade fica bloqueado até conversão.'
    },
    {
        'Problema': 'id_cliente com tipo misto (texto vs. inteiro) entre clientes e respondentes',
        'Indicadores afetados': 'I10, I11',
        'Dashboard afetado': 'D02 — Adesão e representatividade',
        'Risco': 'JOIN entre clientes e respondentes pode retornar zero correspondências ou correspondências incorretas se os tipos não forem normalizados antes.'
    },
    {
        'Problema': 'data_solicitacao ausente em 4 pesquisas',
        'Indicadores afetados': 'I01, I03, I04',
        'Dashboard afetado': 'D01',
        'Risco': 'Essas pesquisas não aparecem em filtros de período, distorcendo o volume real por ano.'
    },
    {
        'Problema': 'Brasil/BRASIL/brasil em empresas.País',
        'Indicadores afetados': 'I07, I09',
        'Dashboard afetado': 'D02',
        'Risco': 'Filtro por país retorna subconjuntos incorretos; contagem de empresas nacionais fica subestimada.'
    },
]

df_impactos = pd.DataFrame(impactos)
print('=== Impacto das inconsistências nos KPIs ===')
for _, r in df_impactos.iterrows():
    print(f"\nProblema: {r['Problema']}")
    print(f"  Indicadores: {r['Indicadores afetados']}")
    print(f"  Dashboard:   {r['Dashboard afetado']}")
    print(f"  Risco:       {r['Risco']}")

In [ ]:
# Gráfico: resumo de completude das variáveis críticas para KPIs
campos_criticos = {
    'pesquisas.complexidade':           100 - 75.4,
    'pesquisas.tempo_preenchimento':    100 - 36.9,
    'pesquisas.objetivos':              100 - 33.6,
    'pesquisas.part_mercado':           100 - 12.0,
    'pesquisas.participantes':          100 - 8.5,
    'pesquisas.data_solicitacao':       100 - 1.1,
    'respondentes.id_cliente':          100 - 4.6,
    'respondentes.market_share':        100 - 0.03,
    'empresas.Nacionalidade':           100 - 8.9,
    'empresas.Cidade':                  100 - 5.4,
}

campos = list(campos_criticos.keys())
valores = list(campos_criticos.values())
cores = ['#e07b39' if v < 90 else '#4a90c4' for v in valores]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(campos, valores, color=cores, height=0.65)
for b, v in zip(bars, valores):
    ax.text(v + 0.3, b.get_y() + b.get_height() / 2,
            f'{v:.1f}%', va='center', fontsize=9)

ax.axvline(90, color='red', linewidth=1, linestyle='--', alpha=0.6, label='Limiar 90%')
ax.set_xlabel('Completude (%)')
ax.set_xlim(0, 110)
ax.set_title('Completude de variáveis críticas para KPIs', fontweight='bold')

p1 = mpatches.Patch(color='#e07b39', label='< 90% — requer atenção')
p2 = mpatches.Patch(color='#4a90c4', label='≥ 90% — aceitável')
ax.legend(handles=[p1, p2], loc='lower right')
plt.tight_layout()
plt.savefig('fig_03_completude_critica.png', bbox_inches='tight')
plt.show()

## Conclusões

### Principais achados

1. **Nenhuma base apresentou duplicatas** de chave primária — ponto positivo para a qualidade estrutural dos dados.

2. **Ausências mais críticas** (a investigar, impacto em KPIs):
   - `complexidade` em `pesquisas.csv`: 75,4% ausente — impede análise de tempo por nível de dificuldade
   - `hora_inicio`/`hora_termino` em `respondentes.csv`: 95,7% ausente — inviabiliza KPI de tempo de resposta
   - `id_cliente` em `respondentes.csv`: 4,6% ausente — reduz a base de análise de engajamento por pessoa

3. **Inconsistências de domínio** que exigem normalização antes da ingestão:
   - Capitalização divergente em `País` e `Associado`
   - Percentuais em texto com vírgula decimal
   - Sentinela `s/i` misturada a IDs
   - Tipos incompatíveis de `id_cliente` entre bases